In [ ]:
# imports

import pandas as pd
import numpy as np
import json
import polars as pl
from pathlib import Path
# There is also auto tokenizer for general purpose but just using BERT for simplicity
from transformers import TrainingArguments, Trainer
from datasets import Dataset, DatasetDict
import torch
import glob
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

c:\Users\aca23lr\source\dissertation\tabfact_bert_classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# # Extract tabfact zip if notebook is in google colab

# import zipfile
# import os

# zip_path = '/content/tabfact_data.zip'
# extract_path = '/content/'

# print("Unzipping dataset...")
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_path)
# print("Unzipping completed! Files extracted to:", extract_path)

In [4]:
# Set data path, if notebook is in google colab use content, if not it is in the directory above

# Local repository path
DATA_PATH = "../tabfact_data"

# # Colab path
# DATA_PATH = "/content/tabfact_data"

In [5]:
# Helpers to read in the json data and convert to dataframe

def load_json_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def convert_to_dataframe(data, is_full_cleaned=False):
    record = []
    for table_id, details in data.items():
        statements = details[0]
        enatiled_statuses = details[1]
        caption = details[-1]
        
        if is_full_cleaned:
            pos_tags = details[2]
            for statement, label, statement_pos_tags in zip(statements, enatiled_statuses, pos_tags):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "label": label,
                    "caption": caption,
                    "pos_tags": statement_pos_tags
                }) 
        else:
            for statement, label in zip(statements, enatiled_statuses):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "label": label,
                    "caption": caption
                })
    return pd.DataFrame(record)

def load_and_convert_data_to_dataframe(path, is_full_cleaned=False):
    raw_data = load_json_data(path)
    
    return convert_to_dataframe(raw_data, is_full_cleaned=is_full_cleaned)

In [ ]:
# Load in table dataframe

# DOESN'T WORK IN COLAB BUT FASTER IN LOCAL REPO
# df_tables = (
#     pl.read_csv(
#         "../tabfact_data/tables/*.csv",
#         has_header=False,
#         separator="\1", # Non-existent delimiter so full line/file stays intact
#         new_columns=["table_content"],
#         include_file_paths="file_name",
#     )
#     .group_by("file_name", maintain_order=True) # Group all the table lines together 
#     .agg(pl.col("table_content").str.join("\n")) # Concatenate table lines together
#     .to_pandas()
# )

csv_files = sorted(glob.glob(f"{DATA_PATH}/tables/*.csv"))

df_tables = (
    pl.concat([
        pl.read_csv(
            file,
            has_header=False,
            separator="\1",
            new_columns=["table_content"],
        ).with_columns(pl.lit(Path(file).name).alias("file_name"))
        for file in csv_files
    ])
    .group_by("file_name", maintain_order=True)  # Group all the table lines together
    .agg(pl.col("table_content").str.join("\n"))  # Concatenate table lines together
    .to_pandas()
)

df_tables["file_name"] = df_tables["file_name"].apply(lambda p: Path(p).name) # Only keep the table id

In [18]:
# Defining column labelled rows
print(df_tables.shape)

df_tables["headers"] = df_tables["table_content"].apply(lambda x: x.split("\n")[0].split("#")) # Get the first line of the table content as headers

df_tables["rows"] = df_tables["table_content"].apply(lambda x: [row.split("#") for row in x.split("\n")[1:]]) # Get the rest of the lines as rows

df_tables["column_labelled_rows"] = df_tables.apply(
    lambda row: "\n".join(
        ", ".join(f"{h}: {v}" for h, v in zip(row["headers"], r))
        for r in row["rows"]
    ),
    axis=1,
)

print(df_tables["column_labelled_rows"].iloc[0])

(16573, 5)
aircraft: robinson r - 22, description: light utility helicopter, max gross weight: 1370 lb (635 kg), total disk area: 497 ft square (46.2 m square), max disk loading: 2.6 lb / ft square (14 kg / m square)
aircraft: bell 206b3 jetranger, description: turboshaft utility helicopter, max gross weight: 3200 lb (1451 kg), total disk area: 872 ft square (81.1 m square), max disk loading: 3.7 lb / ft square (18 kg / m square)
aircraft: ch - 47d chinook, description: tandem rotor helicopter, max gross weight: 50000 lb (22680 kg), total disk area: 5655 ft square (526 m square), max disk loading: 8.8 lb / ft square (43 kg / m square)
aircraft: mil mi - 26, description: heavy - lift helicopter, max gross weight: 123500 lb (56000 kg), total disk area: 8495 ft square (789 m square), max disk loading: 14.5 lb / ft square (71 kg / m square)
aircraft: ch - 53e super stallion, description: heavy - lift helicopter, max gross weight: 73500 lb (33300 kg), total disk area: 4900 ft square (460 m 

In [2]:
# Load in non table data frames

df_full_cleaned = load_and_convert_data_to_dataframe(f"{DATA_PATH}/full_cleaned.json", is_full_cleaned=True)

df_full_cleaned_refuted = df_full_cleaned[df_full_cleaned['label'] == 0]
df_full_cleaned_entailed = df_full_cleaned[df_full_cleaned['label'] == 1]

df_r1_training = load_and_convert_data_to_dataframe(f"{DATA_PATH}/r1_training_all.json")
df_r2_training = load_and_convert_data_to_dataframe(f"{DATA_PATH}/r2_training_all.json")

df_training = load_and_convert_data_to_dataframe(f"{DATA_PATH}/train_examples.json")
df_test = load_and_convert_data_to_dataframe(f"{DATA_PATH}/test_examples.json")
df_validation = load_and_convert_data_to_dataframe(f"{DATA_PATH}/val_examples.json")


NameError: name 'load_and_convert_data_to_dataframe' is not defined

In [ ]:
def get_table_claim_df(df, table_df_col, table_before_claim=True, table_prefix="TABLE: ", claim_prefix="CLAIM: ", table_claim_separator=" | "):
    df_table_claim = pd.merge(
        df_tables[["file_name", table_df_col]],
        df[["table_id", "statement", "label"]],
        left_on="file_name",
        right_on="table_id",
        how="inner"
    ).reset_index(drop=True)

    if table_before_claim:
        df_table_claim["table_claim"] = table_prefix + df_table_claim[table_df_col] + table_claim_separator + claim_prefix + df_table_claim["statement"]
    else:
        df_table_claim["table_claim"] = claim_prefix + df_table_claim["statement"] + table_claim_separator + table_prefix + df_table_claim[table_df_col]

    return df_table_claim[["table_claim", "label"]].copy()

df_train_table_claim = get_table_claim_df(df_training, "table_content", table_before_claim=False)
df_val_table_claim = get_table_claim_df(df_validation, "table_content", table_before_claim=False)
df_test_table_claim = get_table_claim_df(df_test, "table_content", table_before_claim=False)

print(df_train_table_claim["table_claim"].iloc[11])

print(df_train_table_claim.shape)


In [ ]:
# Define BERT model and tokenizer
gpt2_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
gpt2_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
gpt2_tokenizer.save_pretrained("./tabfact_bert_tonizer")

In [ ]:
def tokenize_function(col):
    return gpt2_tokenizer(col['table_claim'], padding='max_length', truncation=True)

# Convert to dataset object for BERT training
dataset_train_table_claim = Dataset.from_pandas(df_train_table_claim)
dataset_val_table_claim = Dataset.from_pandas(df_val_table_claim)

datasets = DatasetDict({
    "train": dataset_train_table_claim,
    "val": dataset_val_table_claim
})

tokenized_datasets = datasets.map(tokenize_function, batched=True)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(y_true=labels, y_pred=predictions)
    f1 = f1_score(y_true=labels, y_pred=predictions)
    precision = precision_score(y_true=labels, y_pred=predictions)
    recall = recall_score(y_true=labels, y_pred=predictions)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [ ]:
# Training the model with corrected metrics

if torch.cuda.is_available():
    print(f"GPU Available: {torch.cuda.get_device_name(0)}")
    # training_args = TrainingArguments(
    #     output_dir='./results',
    #     eval_strategy="epoch",
    #     save_strategy="epoch",
    #     load_best_model_at_end=True,
    #     metric_for_best_model="accuracy",
    #     learning_rate=2e-5,
    #     per_device_train_batch_size=16,
    #     per_device_eval_batch_size=32,
    #     num_train_epochs=3,
    #     weight_decay=0.01,
    #     fp16=True
    # )
    training_args = TrainingArguments(
        output_dir='./results',
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        learning_rate=1e-5, 
        warmup_steps=1730,  # Warms up LR over the first 10% of training steps 
        max_grad_norm=1.0, # Enable gradient clipping to cap wild gradient updates
        seed=42,
        data_seed=42,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        fp16=True
    )

    trainer = Trainer(
        model=gpt2_model,
        args=training_args,
        train_dataset=tokenized_datasets['train'],
        eval_dataset=tokenized_datasets['val'],
        compute_metrics=compute_metrics
    )

    trainer.train()
    trainer.save_model("./tabfact_gpt2")
else:
    print("GPU NOT active. Running on CPU. Training will not commence")

In [ ]:
dataset_test_table_claim = Dataset.from_pandas(df_test_table_claim)

tokenized_test_dataset = dataset_test_table_claim.map(tokenize_function, batched=True)

test_results = trainer.predict(tokenized_test_dataset)

print("Test Set Performance")
for metric_name, metric_value in test_results.metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

In [ ]:
# # Google colab model downloading

# from google.colab import files
# import shutil

# print("Zipping model and tokenizer folders together...")
# # Create a single zip containing the model and the tokenizer
# shutil.make_archive('/content/tabfact_gpt2_complete', 'zip', '/content', 'tabfact_gpt2')
# shutil.make_archive('/content/tabfact_gpt2_tokenizer_complete', 'zip', '/content', 'tabfact_gpt2_tokenizer')

# print("Ready for download!")
# # Trigger downloads
# files.download('/content/tabfact_gpt2_complete.zip')
# files.download('/content/tabfact_gpt2_tokenizer_complete.zip')